In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt

from scipy.optimize import minimize_scalar
from scipy.special import expit, logit
from scipy.stats import norm
import utilities as utils

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class GaussianNetwork(nn.Module):
    """
    Neural network mapping x -> parameters of an N_y-dimensional Gaussian.

    Outputs
    -------
    mean : Tensor, shape (..., N_y)
        Gaussian mean.

    scale_tril : Tensor, shape (..., N_y, N_y)
        Lower-triangular Cholesky factor L such that
            covariance = L @ L.T

    covariance : Tensor, shape (..., N_y, N_y)
        Gaussian covariance matrix.
    """

    def __init__(
        self,
        input_dim,
        output_dim,
        hidden_dims=(128, 128),
        min_std=1e-4,
    ):
        super().__init__()

        self.output_dim = output_dim
        self.min_std = min_std

        # Shared feature-processing network
        layers = []
        d = input_dim

        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(d, hidden_dim),
                nn.ReLU(),
            ])
            d = hidden_dim

        self.backbone = nn.Sequential(*layers)

        # Mean requires N_y parameters
        self.mean_head = nn.Linear(d, output_dim)

        # Lower triangular matrix requires
        # N_y * (N_y + 1) / 2 parameters
        n_tril = output_dim * (output_dim + 1) // 2
        self.cholesky_head = nn.Linear(d, n_tril)

        # Indices of lower-triangular matrix elements
        tril_indices = torch.tril_indices(
            row=output_dim,
            col=output_dim,
            offset=0,
        )

        self.register_buffer("tril_indices", tril_indices)


    def forward(self, x):
        h = self.backbone(x)

        # Mean
        mean = self.mean_head(h)

        # Raw parameters for Cholesky factor
        raw_tril = self.cholesky_head(h)

        batch_shape = x.shape[:-1]

        L = torch.zeros(
            *batch_shape,
            self.output_dim,
            self.output_dim,
            device=x.device,
            dtype=x.dtype,
        )

        L[..., self.tril_indices[0], self.tril_indices[1]] = raw_tril

        # Diagonal must be positive.
        diag_idx = torch.arange(self.output_dim, device=x.device)

        raw_diag = L[..., diag_idx, diag_idx]

        L[..., diag_idx, diag_idx] = (
            F.softplus(raw_diag) + self.min_std
        )

        covariance = L @ L.transpose(-1, -2)

        return mean, L, covariance

In [5]:
def generate_trials(n_trials, mirror_trials=False, balanced_starts=False, seed=None):
    """Generate trials from a two-state switching Gaussian process.

    Parameters
    ----------
    n_trials : int
        Number of independent trials to generate.
    balanced_starts : bool, default=False
        If True, exactly half of the trials start in each source
        distribution. This requires an even ``n_trials``.
    seed : int or None, default=None
        Seed for NumPy's random-number generator. Use an integer for
        reproducible simulations.

    Returns
    -------
    X : ndarray of int, shape (n_trials, 12)
        Clipped and rounded observations for each trial.
    Y : ndarray of int, shape (n_trials,)
        Identity of the source for the final draw: 0 denotes the Gaussian
        with mean -17 and 1 denotes the Gaussian with mean 17.
    """
    if isinstance(n_trials, (bool, np.bool_)) or not isinstance(n_trials, (int, np.integer)):
        raise TypeError("n_trials must be an integer")
    if n_trials < 0:
        raise ValueError("n_trials must be non-negative")
    if (balanced_starts or mirror_trials) and n_trials % 2:
        raise ValueError("balanced_starts=True or mirror_trials=True requires an even n_trials")

    rng = np.random.default_rng(seed)
    n_draws = 12
    source_means = np.array([-17.0, 17.0])

    if mirror_trials:
        n_trials_ = n_trials // 2
    else:
        n_trials_ = n_trials

    if balanced_starts:
        starts = np.repeat([0, 1], n_trials_ // 2)
        rng.shuffle(starts)
    else:
        starts = rng.integers(0, 2, size=n_trials_)

    # switches[:, j] indicates whether the source changes between draws
    # j and j + 1. Cumulative parity therefore gives the source at each draw.
    switches = rng.random((n_trials_, n_draws - 1)) < 0.08
    source_paths = np.column_stack(
        [starts, starts[:, None] ^ np.logical_xor.accumulate(switches, axis=1)]
    ).astype(int)

    raw_draws = rng.normal(loc=source_means[source_paths], scale=29.0)
    X = np.rint(np.clip(raw_draws, -90, 90)).astype(int)
    Y = source_paths[:, -1].copy()
    if mirror_trials:
        X = np.concatenate([X, -X], axis=0)
        Y = np.concatenate([Y, 1-Y], axis=0)
    return X, Y

In [6]:
Xsample,Ysample = generate_trials(100000, mirror_trials=True, balanced_starts=True, seed=234)

In [21]:
test_enc = GaussianNetwork(input_dim=12, output_dim=2, hidden_dims=(128, 128), min_std=1e-4)
test_enc.forward(torch.from_numpy(Xsample[:10]).float())[2]

tensor([[[ 3.1208e+00, -3.7374e+00],
         [-3.7374e+00,  5.8293e+00]],

        [[ 2.2475e-02, -1.1427e+00],
         [-1.1427e+00,  6.4222e+01]],

        [[ 9.7438e-01, -1.8955e+00],
         [-1.8955e+00,  2.1737e+01]],

        [[ 1.1015e-01,  5.2301e-01],
         [ 5.2301e-01,  5.5496e+00]],

        [[ 1.2464e+00, -9.8846e+00],
         [-9.8846e+00,  8.0870e+01]],

        [[ 2.7286e-01, -3.5214e+00],
         [-3.5214e+00,  5.3400e+01]],

        [[ 8.3403e+00, -1.2852e+01],
         [-1.2852e+01,  2.8568e+01]],

        [[ 2.5702e-01,  2.2207e+00],
         [ 2.2207e+00,  3.4791e+01]],

        [[ 2.9013e-02, -7.2904e-03],
         [-7.2904e-03,  1.6729e+00]],

        [[ 2.5448e+00, -3.6986e+00],
         [-3.6986e+00,  8.1185e+00]]], grad_fn=<UnsafeViewBackward0>)

In [ ]:
def gaussian_kl_divergence_torch(
    true_mean,
    true_scale_tril,
    approx_mean,
    approx_scale_tril,
):
    true_dist = torch.distributions.MultivariateNormal(
        loc=true_mean,
        scale_tril=true_scale_tril,
    )

    approx_dist = torch.distributions.MultivariateNormal(
        loc=approx_mean,
        scale_tril=approx_scale_tril,
    )

    return torch.distributions.kl_divergence(
        true_dist,
        approx_dist,
    )

def p_RgX_mc_est(mean,L,n_samples=1000):
    """Monte Carlo estimate of p(R|X) for a Gaussian encoder.

    Parameters
    ----------
    mean : Tensor, shape (n_trials, N_y)
        Mean of the Gaussian encoder.
    L : Tensor, shape (n_trials, N_y, N_y)
        Lower-triangular Cholesky factor of the covariance of the Gaussian
        encoder.
    n_samples : int, default=1000
        Number of Monte Carlo samples to use for the estimate."""

    p_ZgX = torch.distributions.MultivariateNormal(
        loc=mean,
        scale_tril=L,
    )

    ZgX_samples = p_ZgX.rsample((n_samples,))
    R_samples = torch.argmax(ZgX_samples, dim=(-1, -2))
    R_onehot = F.one_hot(R_samples, num_classes=mean.shape[-1])
    p_RgX = R_onehot.float().mean(dim=-1)
    return p_RgX

def p_RgY_mc_est(mean,L,Y,n_samples=1000):
    """Monte Carlo estimate of p(R|Y) for a Gaussian encoder.

    Parameters
    ----------
    mean : Tensor, shape (n_trials, N_y)
        Mean of the Gaussian encoder.
    L : Tensor, shape (n_trials, N_y, N_y)
        Lower-triangular Cholesky factor of the covariance of the Gaussian
        encoder.
    Y : Tensor, shape (n_trials,)
        Ground-truth source identity for each trial.
    n_samples : int, default=1000
        Number of Monte Carlo samples to use for the estimate."""

    p_RgX = p_RgX_mc_est(mean,L,n_samples=n_samples)
    p_RgY = torch.zeros((mean.shape[-1],mean.shape[-1],), device=mean.device, dtype=mean.dtype)
    Y_onehot = F.one_hot(Y, num_classes=mean.shape[-1])
    p_RgY = p_RgX.T @ Y_onehot.float() / Y_onehot.float().sum(dim=0, keepdim=True)

    return p_RgY

def I_RY(mean,L,Y,n_samples=1000):
    """Monte Carlo estimate of I(R;Y) for a Gaussian encoder.

    Parameters
    ----------
    mean : Tensor, shape (n_trials, N_y)
        Mean of the Gaussian encoder.
    L : Tensor, shape (n_trials, N_y, N_y)
        Lower-triangular Cholesky factor of the covariance of the Gaussian
        encoder.
    Y : Tensor, shape (n_trials,)
        Ground-truth source identity for each trial.
    n_samples : int, default=1000
        Number of Monte Carlo samples to use for the estimate."""

    p_RgY = p_RgY_mc_est(mean,L,Y,n_samples=n_samples)
    p_Y = F.one_hot(Y, num_classes=mean.shape[-1]).float().mean(dim=0,keepdim=True)
    p_R = (p_RgY * p_Y).sum(dim=1, keepdim=True)
    I_RY = (p_RgY * p_Y * (p_RgY / p_R).log()).sum(dim=1, keepdim=True).sum()
    return I_RY

def variational_obj(beta,mean,L,Y,n_samples=1000):
    """Monte Carlo estimate of the variational objective for a Gaussian encoder.

    Parameters
    ----------
    beta : float
        Trade-off parameter between I(X;R) and I(R;Y).

    mean : Tensor, shape (n_trials, N_y)
        Mean of the Gaussian encoder.
    L : Tensor, shape (n_trials, N_y, N_y)
        Lower-triangular Cholesky factor of the covariance of the Gaussian
        encoder.
    Y : Tensor, shape (n_trials,)
        Ground-truth source identity for each trial.
    n_samples : int, default=1000
        Number of Monte Carlo samples to use for the estimate."""

    DKL_ZgX = gaussian_kl_divergence_torch(
        true_mean=mean,
        true_scale_tril=L,
        approx_mean=torch.zeros_like(mean, device=mean.device, dtype=mean.dtype),
        approx_scale_tril=torch.eye(mean.shape[-1], device=mean.device, dtype=mean.dtype).expand(mean.shape[0], -1, -1),
    )

    I_XR_est = DKL_ZgX.mean()
    I_RY_est = I_RY(mean,L,Y,n_samples=n_samples)
    return I_XR_est - beta * I_RY_est